# Initialization and Data Preparation

### Set Global Seed & Load .csv Data

In [6]:
from pathlib import Path

import pandas as pd

### Set the Global Random Seed for Reproducibility ###
seed = 42

### Load the Data ###
data_dir = Path("Datasets") # Change the path to your data folder directory

train_df = pd.read_csv(data_dir / "loan_data_smoted_raw_train_4.csv")
test_df = pd.read_csv(data_dir / "loan_data_raw_test_1.csv")



### Data Sanity Checks

In [7]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTrain columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())
print("\nTrain missing values:")
print(train_df.isna().sum())
print("\nTest missing values:")
print(test_df.isna().sum())
print("\nTrain duplicate rows:", train_df.duplicated().sum())
print("Test duplicate rows:", test_df.duplicated().sum())

target_col = next((col for col in ["loan_status", "target", "class", "label", "y"] if col in train_df.columns), None)
if target_col:
    print(f"\nTarget column: {target_col}")
    print("Train target distribution:")
    print(train_df[target_col].value_counts(dropna=False))
    print("\nTest target distribution:")
    print(test_df[target_col].value_counts(dropna=False))

Train shape: (53676, 13)
Test shape: (8999, 13)

Train columns: ['person_education', 'person_age', 'person_gender', 'person_income', 'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'loan_status']
Test columns: ['person_education', 'person_age', 'person_gender', 'person_income', 'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'loan_status']

Train missing values:
person_education              0
person_age                    0
person_gender                 0
person_income                 0
person_emp_exp                0
person_home_ownership         0
loan_amnt                     0
loan_intent                   0
loan_int_rate                 0
loan_percent_income           0
cb_person_cred_hist_length    0
credit_score                  0
loan_status                   0
d

### Create Training and Testing Data Matrices and Reponse Vector

In [8]:
response_col = "loan_status"

X_train = train_df.drop(columns=[response_col])
y_train = train_df[response_col]

X_test = test_df.drop(columns=[response_col])
y_test = test_df[response_col]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (53676, 12)
y_train shape: (53676,)
X_test shape: (8999, 12)
y_test shape: (8999,)


### One-hot Encoding

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_cols = X_train.select_dtypes(include=["object", "category", "string"]).columns.tolist()
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

Categorical columns: ['person_gender', 'person_home_ownership', 'loan_intent']
Numeric columns: ['person_education', 'person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']


# XGBoost Implementation

### XGBoost Tuning

In [10]:
from xgboost import XGBClassifier
from scipy.stats import randint, uniform
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

# Set up the XGBoost model and pipeline
xgb_model = XGBClassifier(
    objective="binary:logistic",                    # default objective for binary classification
    eval_metric="logloss",                          # default evaluation metric for binary classification
    tree_method="hist",
    random_state=seed,
    n_jobs=-1,
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", xgb_model),
    ]
)

# Define the hyperparameter search space for RandomizedSearchCV
param_distributions = {
    "model__n_estimators": randint(300, 550),
    "model__max_depth": randint(6, 10),
    "model__learning_rate": uniform(0.07, 0.09),
    "model__subsample": uniform(0.78, 0.17),
    "model__colsample_bytree": uniform(0.75, 0.20),
    "model__min_child_weight": randint(3, 8),
    "model__gamma": uniform(0.05, 0.35),
    "model__reg_lambda": uniform(1.5, 2.0),
    "model__reg_alpha": uniform(0, 0.5),
}

# Set up Stratified K-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

# Perform RandomizedSearchCV to find the best hyperparameters
xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_distributions,
    n_iter=40,
    scoring="f1",                               # Random Search chooses the best hyperparameters based on the F1 score
    cv=cv,
    random_state=seed,
    n_jobs=-1,
    verbose=1,
)

# Fit the model and evaluate performance
xgb_search.fit(X_train, y_train)

best_xgb_model = xgb_search.best_estimator_
y_proba = best_xgb_model.predict_proba(X_test)[:, 1]
y_pred = best_xgb_model.predict(X_test)

# Calculate confusion matrix and by-class accuracy
cm = confusion_matrix(y_test, y_pred)                           # Confusion matrix
by_class_accuracy = cm.diagonal() / cm.sum(axis=1)                  

print("Best CV F1 score:", xgb_search.best_score_)
print("Best parameters:")
print(xgb_search.best_params_)
print("\nConfusion matrix: Row: Actual class, Column: Predicted class")
print(cm)
print("\nBy-class accuracy:")
for class_label, accuracy in zip(best_xgb_model.classes_, by_class_accuracy):
    print(f"Class {class_label}: {accuracy:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best CV F1 score: 0.9432947215073553
Best parameters:
{'model__colsample_bytree': np.float64(0.8273470692601075), 'model__gamma': np.float64(0.37785549605785707), 'model__learning_rate': np.float64(0.0823768849731394), 'model__max_depth': 9, 'model__min_child_weight': 3, 'model__n_estimators': 516, 'model__reg_alpha': np.float64(0.10453581036885684), 'model__reg_lambda': np.float64(2.582895947655132), 'model__subsample': np.float64(0.898283347888664)}

Confusion matrix: Row: Actual class, Column: Predicted class
[[6806  193]
 [ 576 1424]]

By-class accuracy:
Class 0: 0.9724
Class 1: 0.7120

Classification report:
              precision    recall  f1-score   support

           0       0.92      0.97      0.95      6999
           1       0.88      0.71      0.79      2000

    accuracy                           0.91      8999
   macro avg       0.90      0.84      0.87      8999
weighted avg       0.91      0.91      0.91  